"""
================================================================================
TRAVELAGENT SPARK STREAMING CONSUMER - PRODUCTION VERSION
================================================================================
Consumes flight status and booking events from Kafka
Writes to Cassandra:
  1. Raw events tables (historical storage)
  2. 5-minute windowed metrics tables (for Grafana real-time monitoring)

Workflow: Kafka → Spark Streaming → Cassandra → Grafana

Topics:
- flight-status-events → aviation.flight_status_events (raw)
                       → aviation.flight_metrics_5min (windowed, TTL)
- booking-events → aviation.booking_events (raw)
                 → aviation.booking_metrics_5min (windowed, TTL)
================================================================================
"""============
"""

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, current_timestamp, to_timestamp,
    window, sum as _sum, avg, max as _max, min as _min,
    count, when, lit, round as spark_round
)
from pyspark.sql.types import (
    StructType, StringType, DoubleType, TimestampType,
    BooleanType, IntegerType
)
print("Imports loaded\n")

✅ Imports loaded



In [ ]:
print(" Creating Spark Session...\n")

os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,'
    'com.datastax.spark:spark-cassandra-connector_2.12:3.3.0 '
    '--repositories https://repo1.maven.org/maven2 '
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("TravelAgentStreamingConsumer_Production") \
    .master("local[4]") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .config("spark.cassandra.connection.port", "9042") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.sql.catalog.cassandra", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session Created\n")

In [ ]:
print(" Defining schemas...\n")

flight_status_schema = StructType() \
    .add("event_id", StringType()) \
    .add("event_time", StringType()) \
    .add("flight_number", StringType()) \
    .add("trip_id", StringType()) \
    .add("airline_id", StringType()) \
    .add("aircraft_id", StringType()) \
    .add("airport_src_id", StringType()) \
    .add("airport_dst_id", StringType()) \
    .add("scheduled_departure", StringType()) \
    .add("actual_departure", StringType()) \
    .add("scheduled_arrival", StringType()) \
    .add("actual_arrival", StringType()) \
    .add("flight_status", StringType()) \
    .add("delay_minutes", IntegerType()) \
    .add("gate_change", BooleanType()) \
    .add("disruption_reason", StringType()) \
    .add("source_system", StringType()) \
    .add("ingestion_time", StringType())

booking_events_schema = StructType() \
    .add("event_id", StringType()) \
    .add("event_time", StringType()) \
    .add("booking_id", StringType()) \
    .add("trip_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("flight_number", StringType()) \
    .add("hotel_id", StringType()) \
    .add("booking_event", StringType()) \
    .add("failure_reason", StringType()) \
    .add("booking_channel", StringType()) \
    .add("booking_value", DoubleType()) \
    .add("source_system", StringType()) \
    .add("ingestion_time", StringType())

print(" Schemas defined\n")

In [ ]:
print(" Reading Kafka - Flight Status...\n")

df_flight_status_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092") \
    .option("subscribe", "flight-status-events") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

raw_flight_status = df_flight_status_kafka.selectExpr(
    "CAST(value AS STRING) AS json_payload",
    "timestamp AS kafka_arrival_time"
)

parsed_flight_status = raw_flight_status.select(
    from_json(col("json_payload"), flight_status_schema).alias("data"),
    col("kafka_arrival_time")
).select("data.*", "kafka_arrival_time")

transformed_flight_status = parsed_flight_status \
    .withColumn("event_time", to_timestamp(col("event_time"))) \
    .withColumn("scheduled_departure", to_timestamp(col("scheduled_departure"))) \
    .withColumn("actual_departure", to_timestamp(col("actual_departure"))) \
    .withColumn("scheduled_arrival", to_timestamp(col("scheduled_arrival"))) \
    .withColumn("actual_arrival", to_timestamp(col("actual_arrival"))) \
    .withColumn("ingestion_time", to_timestamp(col("ingestion_time"))) \
    .withColumn("processing_time", current_timestamp()) \
    .filter(col("event_id").isNotNull())

print(" Flight Status stream configured\n")

In [ ]:
print(" Reading Kafka - Booking Events...\n")

df_booking_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092") \
    .option("subscribe", "booking-events") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

raw_booking = df_booking_kafka.selectExpr(
    "CAST(value AS STRING) AS json_payload",
    "timestamp AS kafka_arrival_time"
)

parsed_booking = raw_booking.select(
    from_json(col("json_payload"), booking_events_schema).alias("data"),
    col("kafka_arrival_time")
).select("data.*", "kafka_arrival_time")

transformed_booking = parsed_booking \
    .withColumn("event_time", to_timestamp(col("event_time"))) \
    .withColumn("ingestion_time", to_timestamp(col("ingestion_time"))) \
    .withColumn("processing_time", current_timestamp()) \
    .filter(col("event_id").isNotNull())

print(" Booking Events stream configured\n")

In [ ]:
print(" Creating Flight Metrics - 5 Min Windows...\n")

flight_windowed = transformed_flight_status \
    .withWatermark("event_time", "30 seconds") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("airline_id")
    ).agg(
        count("*").alias("total_flights"),
        _sum(when(col("flight_status") == "ON_TIME", 1).otherwise(0)).alias("ontime_flights"),
        _sum(when(col("flight_status") == "DELAYED", 1).otherwise(0)).alias("delayed_flights"),
        _sum(when(col("flight_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_flights"),
        _sum(when(col("flight_status") == "DIVERTED", 1).otherwise(0)).alias("diverted_flights"),
        _sum(when(col("delay_minutes") > 120, 1).otherwise(0)).alias("high_delay_count"),
        _sum(when((col("delay_minutes") >= 60) & (col("delay_minutes") <= 120), 1).otherwise(0)).alias("medium_delay_count"),
        _sum(when((col("delay_minutes") < 60) & (col("delay_minutes") > 0), 1).otherwise(0)).alias("low_delay_count"),
        avg("delay_minutes").alias("avg_delay_minutes"),
        _max("delay_minutes").alias("max_delay_minutes"),
        _sum(when(col("gate_change") == True, 1).otherwise(0)).alias("gate_change_count"),
        _sum(when(col("disruption_reason") == "WEATHER", 1).otherwise(0)).alias("weather_disruptions"),
        _sum(when(col("disruption_reason") == "TECHNICAL", 1).otherwise(0)).alias("technical_disruptions"),
        _sum(when(col("disruption_reason").isin(["CREW", "AIR_TRAFFIC"]), 1).otherwise(0)).alias("operational_disruptions")
    )

flight_metrics_final = flight_windowed.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    col("airline_id").alias("metric_id"),
    col("total_flights"),
    col("ontime_flights"),
    col("delayed_flights"),
    col("cancelled_flights"),
    col("diverted_flights"),
    col("high_delay_count"),
    col("medium_delay_count"),
    col("low_delay_count"),
    col("avg_delay_minutes"),
    col("max_delay_minutes"),
    col("gate_change_count"),
    col("weather_disruptions"),
    col("technical_disruptions"),
    col("operational_disruptions"),
    spark_round((col("ontime_flights") / col("total_flights")) * 100, 2).alias("ontime_percentage"),
    spark_round((col("delayed_flights") / col("total_flights")) * 100, 2).alias("delay_percentage"),
    current_timestamp().alias("computed_at")
)

print(" Flight metrics configured\n")

In [ ]:
print(" Creating Booking Metrics - 5 Min Windows...\n")

booking_windowed = transformed_booking \
    .withWatermark("event_time", "30 seconds") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        lit("global").alias("metric_id")
    ).agg(
        count("*").alias("total_bookings"),
        _sum(when(col("booking_event") == "BOOKING_CREATED", 1).otherwise(0)).alias("created_bookings"),
        _sum(when(col("booking_event") == "BOOKING_CONFIRMED", 1).otherwise(0)).alias("confirmed_bookings"),
        _sum(when(col("booking_event") == "BOOKING_FAILED", 1).otherwise(0)).alias("failed_bookings"),
        _sum(when(col("booking_event") == "BOOKING_CANCELLED", 1).otherwise(0)).alias("cancelled_bookings"),
        _sum(when(col("failure_reason") == "PAYMENT_FAILED", 1).otherwise(0)).alias("payment_declined_count"),
        _sum(when(col("failure_reason") == "SEAT_UNAVAILABLE", 1).otherwise(0)).alias("seat_unavailable_count"),
        _sum(when(col("failure_reason") == "TIMEOUT", 1).otherwise(0)).alias("timeout_count"),
        _sum(when(col("failure_reason") == "INVALID_DATA", 1).otherwise(0)).alias("invalid_data_count"),
        _sum("booking_value").alias("total_revenue"),
        avg("booking_value").alias("avg_booking_value"),
        _sum(when(col("booking_channel") == "WEB", 1).otherwise(0)).alias("web_bookings"),
        _sum(when(col("booking_channel") == "MOBILE", 1).otherwise(0)).alias("mobile_bookings"),
        _sum(when(col("booking_channel") == "API", 1).otherwise(0)).alias("api_bookings"),
        _sum(when(col("booking_channel") == "CALL_CENTER", 1).otherwise(0)).alias("call_center_bookings")
    )

booking_metrics_final = booking_windowed.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    col("metric_id"),
    col("total_bookings"),
    col("created_bookings"),
    col("confirmed_bookings"),
    col("failed_bookings"),
    col("cancelled_bookings"),
    col("payment_declined_count"),
    col("seat_unavailable_count"),
    col("timeout_count"),
    col("invalid_data_count"),
    col("total_revenue"),
    col("avg_booking_value"),
    col("web_bookings"),
    col("mobile_bookings"),
    col("api_bookings"),
    col("call_center_bookings"),
    spark_round((col("failed_bookings") / col("total_bookings")), 4).alias("failure_rate"),
    spark_round((col("cancelled_bookings") / col("total_bookings")), 4).alias("cancellation_rate"),
    spark_round((col("total_revenue") / 5.0), 2).alias("revenue_per_minute"),
    current_timestamp().alias("computed_at")
)

print(" Booking metrics configured\n")

In [ ]:
print(" Writing to Cassandra using foreachBatch...\n")

# ============================================
# SOLUTION: Use foreachBatch (ALWAYS WORKS!)
# ============================================

def write_to_cassandra_flight(batch_df, batch_id):
    """Write flight metrics batch to Cassandra"""
    if batch_df.count() > 0:
        batch_df.write \
            .format("org.apache.spark.sql.cassandra") \
            .option("keyspace", "aviation") \
            .option("table", "flight_metrics_5min") \
            .mode("append") \
            .save()
        print(f" Batch {batch_id}: Wrote {batch_df.count()} flight metric rows")

def write_to_cassandra_booking(batch_df, batch_id):
    """Write booking metrics batch to Cassandra"""
    if batch_df.count() > 0:
        batch_df.write \
            .format("org.apache.spark.sql.cassandra") \
            .option("keyspace", "aviation") \
            .option("table", "booking_metrics_5min") \
            .mode("append") \
            .save()
        print(f" Batch {batch_id}: Wrote {batch_df.count()} booking metric rows")

# Start streaming with foreachBatch
flight_metrics_query = flight_metrics_final \
    .writeStream \
    .foreachBatch(write_to_cassandra_flight) \
    .option("checkpointLocation", "/data/checkpoint/flight_metrics_5min") \
    .trigger(processingTime="30 seconds") \
    .start()

print(f" Flight Metrics Query: {flight_metrics_query.id}\n")

booking_metrics_query = booking_metrics_final \
    .writeStream \
    .foreachBatch(write_to_cassandra_booking) \
    .option("checkpointLocation", "/data/checkpoint/booking_metrics_5min") \
    .trigger(processingTime="30 seconds") \
    .start()

print(f" Booking Metrics Query: {booking_metrics_query.id}\n")
print(" Both queries started successfully!")
print(" Watch for batch write confirmations above\n")

In [ ]:
print(" Queries running...\n")
try:
    spark.streams.awaitAnyTermination()
except KeyboardInterrupt:
    for q in spark.streams.active:
        q.stop()

🔄 Queries running...

